# Notebook 05 — end-to-end causal pipeline from a live GraphDB endpoint

Runs the whole pipeline — **causal graph discovery → causal model learning → conditional /
interventional / counterfactual inference** — sourced entirely from a live SPARQL endpoint
(`http://localhost:7200/repositories/syn_clinic`) instead of a local TTL file.
Notebooks 02 and 04 do the same two halves (discovery, then modeling) against
`kgs/ttls/synthetic_clinic.ttl`; this notebook redoes both against the endpoint directly, using
`causalway.sources.resolve_schema` and `CausalModel.fit(..., kg=ENDPOINT)` so the pipeline never
touches a local file.

Run with the `rdfenv` kernel. Needs the GraphDB repository above to be reachable — the first cell
fails fast and explains why if it isn't.

In [1]:
import os, sys

def _find_root():
    p = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(p, 'causalway')) and os.path.isdir(os.path.join(p, 'algs')):
            return p
        p = os.path.dirname(p)
    raise RuntimeError('project root not found')

PROJECT_ROOT = _find_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import dowhy, pgmpy, sklearn, rdflib

print('dowhy', dowhy.__version__, '| pgmpy', pgmpy.__version__, '| sklearn', sklearn.__version__, '| rdflib', rdflib.__version__)
print('PROJECT_ROOT =', PROJECT_ROOT)


dowhy 0.14 | pgmpy 1.1.0 | sklearn 1.6.1 | rdflib 7.6.0
PROJECT_ROOT = /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api


## 1. Connect to the endpoint and resolve its schema

`causalway.sources.resolve_schema` accepts a file path, an `rdflib.Graph`, or (as here) an
`http(s)://` SPARQL endpoint URL — everything downstream (`build_nodes`, `EdgeConstraint`,
`materialize`, `CausalModel.fit`) is written against that one polymorphic entry point, so nothing
below needs to know the KG lives in GraphDB rather than on disk. `infer_missing` defaults to
`False` for endpoints (an A-Box scan for missing T-Box declarations is fine on a file, hostile to
a shared SPARQL endpoint) — pass `infer_missing=True` explicitly if this repository has no T-Box.

In [2]:
from causalway.sources import resolve_schema

ENDPOINT = "http://localhost:7200/repositories/syn_clinic"

try:
    schema_probe = resolve_schema(ENDPOINT, timeout=10)
except Exception as e:
    raise RuntimeError(
        f"Cannot reach {ENDPOINT} ({type(e).__name__}: {e}). This notebook needs the live "
        "GraphDB repository -- start it / fix the URL, then re-run."
    ) from e

print('endpoint reachable:', ENDPOINT)
print(schema_probe.summary())


resolve_schema: infer_missing defaults to False for endpoints (pass infer_missing=True to run the T-Box inference queries against it anyway).


endpoint reachable: http://localhost:7200/repositories/syn_clinic
{'n_classes': 5, 'n_object_properties': 2, 'n_object_domain_range': 2, 'n_data_properties': 11, 'n_data_domain_range': 11, 'n_inferred': 0}


## 2. Assemble the discovery context from the endpoint

`runners.run_kg_discovery.build_context` hard-codes `OntologySchema.from_file`, so it only takes a
local path. `build_context_from_source` below is the same assembly — nodes, `EdgeConstraint`,
flat-join materialization, constant-column drop, discrete/continuous frames — built on
`resolve_schema` instead, so it works for a file, an `rdflib.Graph`, or an endpoint alike.

In [3]:
from causalway.nodes import build_nodes
from causalway.constraints import EdgeConstraint
from causalway.bgp import materialize
from causalway.encoding import drop_constant_columns, to_discrete_frame, to_numeric_frame
from runners.run_kg_discovery import DiscoveryContext, run_algorithm, to_ocg


def build_context_from_source(source, *, infer_missing=None, include_object_properties=True,
                              object_value='range_class', key_properties=None,
                              relation_direction='both', allow_epsilon=True, max_hops=1,
                              optional_data_properties=False, limit=None) -> DiscoveryContext:
    # Same assembly as runners.run_kg_discovery.build_context, generalised from a file
    # path to any causalway.sources.GraphSource (file / rdflib.Graph / endpoint URL).
    schema = resolve_schema(source, infer_missing=infer_missing)
    nodes = build_nodes(schema, include_object_properties=include_object_properties)
    constraint = EdgeConstraint.from_schema(
        schema, nodes, relation_direction=relation_direction,
        allow_epsilon=allow_epsilon, max_hops=max_hops,
    )
    mat = materialize(
        schema, nodes, include_object_properties=include_object_properties,
        object_value=object_value, key_properties=key_properties,
        optional_data_properties=optional_data_properties, limit=limit,
    )
    if isinstance(mat, list):
        raise ValueError(
            'The KG has multiple connected components; materialize and run each '
            'component separately (cross-component edges are forbidden anyway).'
        )

    df = mat.df.reindex(columns=[n.name for n in nodes])
    df_clean, dropped = drop_constant_columns(df, verbose=True)
    keep = [i for i, n in enumerate(nodes) if n.name in df_clean.columns]
    constraint_kept = constraint.subset(keep)
    nodes_kept = [nodes[i] for i in keep]

    return DiscoveryContext(
        schema=schema, nodes=nodes, constraint=constraint, mat=mat,
        nodes_kept=nodes_kept, constraint_kept=constraint_kept,
        discrete_df=to_discrete_frame(df_clean), continuous_df=to_numeric_frame(df_clean),
        dropped_columns=dropped, source=str(source),
    )


ctx = build_context_from_source(ENDPOINT)
print('schema summary:', ctx.schema.summary())
print('nodes (kept):', ctx.column_names)
print('dropped (constant) columns:', ctx.dropped_columns)
print('constraint stats:', ctx.constraint_kept.stats())
print('multiplicity:', ctx.mat.multiplicity)
print('flat-join rows:', len(ctx.discrete_df), '| columns:', len(ctx.discrete_df.columns))


2026-09-23 10:24:07,556 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/castle/backend/__init__.py[line:36] - INFO: You can use `os.environ['CASTLE_BACKEND'] = backend` to set the backend(`pytorch` or `mindspore`).


2026-09-23 10:24:07,569 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/castle/algorithms/__init__.py[line:36] - INFO: You are using ``pytorch`` as the backend.


/Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


resolve_schema: infer_missing defaults to False for endpoints (pass infer_missing=True to run the T-Box inference queries against it anyway).


Dropped constant columns: ['Patient.receives', 'Patient.treatedAt']
schema summary: {'n_classes': 5, 'n_object_properties': 2, 'n_object_domain_range': 2, 'n_data_properties': 11, 'n_data_domain_range': 11, 'n_inferred': 0}
nodes (kept): ['Hospital.airPollution', 'Hospital.careQuality', 'Hospital.region', 'Patient.age', 'Patient.geneticRisk', 'Patient.smoking', 'Patient.survival', 'Patient.tumorStage', 'Therapy.dosage', 'Therapy.drugClass', 'Therapy.toxicity']
dropped (constant) columns: ['Patient.receives', 'Patient.treatedAt']
constraint stats: {'n_nodes': 11, 'n_allowed': 92, 'n_forbidden': 18, 'pruning_rate': 0.16363636363636364}
multiplicity: {'Hospital': 249.2, 'Patient': 2.492, 'Therapy': 1.0}
flat-join rows: 4984 | columns: 11


## 2b. Meta info for the LLM-prior method (`GES-Prior`)

`GES-Prior` (§3 below) asks an LLM to score each variable pair and needs two natural-language
inputs — `var_meta` (what each column *means*) and `pair_meta` (how two columns are *related*).
`causalway.llm_meta.build_var_meta` / `build_pair_meta` already derive both from this KG's own
`rdfs:label` / `rdfs:comment` annotations; the cells below add a second, complementary source:
context pulled from an *external* general-purpose KG endpoint (Wikidata by default) via a partial
mapping from column name to entity id there.

Two ways to supply meta info, both handled by `causalway.kg_endpoint_meta.build_kg_meta`:

1. **Hand-written natural-language text** — `VAR_TEXT_META` / `PAIR_TEXT_META` below, keyed the
  same way as `var_meta` / `pair_meta` (`{column: text}` / `{(col_i, col_j): text}`). Anything
   given here is used verbatim, overriding the KG lookup for that column/pair.
2. **A SPARQL endpoint + entity mapping** — `ENTITY_MAP` maps a subset of columns to entity ids on
  `endpoint` (`https://query.wikidata.org` here). For each mapped column, its 1-hop neighborhood
   (predicate label -> object label) is fetched and summarized by an LLM into `var_meta[column]`.
   For each mapped pair, a relational path within 1 hop (a direct edge, or a shared 1-hop
   neighbor) is looked up and summarized into `pair_meta[(col_i, col_j)]`; pairs with no such path
   get `""`, same as `causalway.llm_meta.build_pair_meta` for topologically-forbidden pairs.

This needs an LLM key (see `algs/.env`) and network access to the endpoint; both are optional —
`GES-Prior` below falls back to no priors (`var_meta=pair_meta=None`) if either is unavailable.

In [4]:
# Partial mapping: discovery column -> Wikidata entity id (requirement (2) of the meta info).
ENTITY_MAP = {
    'Hospital.airPollution': "Q131123",
    'Hospital.careQuality': "Q17003063",
    'Hospital.region': "Q82794",
    'Patient.age': "Q185836",
    'Patient.geneticRisk': "Q7187",
    'Patient.smoking': "Q67433631",
    'Patient.tumorStage': "Q12078",
    'Therapy.drugClass': "Q2585617",
    'Therapy.toxicity': "Q274160",
}

# Optional hand-written natural-language overrides (requirement (1)) -- keyed like
# var_meta / pair_meta; anything set here skips the KG lookup for that column/pair.
VAR_TEXT_META = {}
PAIR_TEXT_META = {}

ENTITY_MAP

{'Hospital.airPollution': 'Q131123',
 'Hospital.careQuality': 'Q17003063',
 'Hospital.region': 'Q82794',
 'Patient.age': 'Q185836',
 'Patient.geneticRisk': 'Q7187',
 'Patient.smoking': 'Q67433631',
 'Patient.tumorStage': 'Q12078',
 'Therapy.drugClass': 'Q2585617',
 'Therapy.toxicity': 'Q274160'}

In [5]:
from causalway.kg_endpoint_meta import build_kg_meta, WIKIDATA_ENDPOINT
from causalway.llm_meta import build_domain_str

domain = build_domain_str(ctx.schema)

try:
    var_meta, pair_meta = build_kg_meta(
        ENTITY_MAP, var_text=VAR_TEXT_META, pair_text=PAIR_TEXT_META,
        endpoint=WIKIDATA_ENDPOINT, llm_model='deepseek-v4-flash',
    )
    print(f"var_meta: {len(var_meta)} columns | pair_meta: {len(pair_meta)} pairs "
          f"(from {WIKIDATA_ENDPOINT})")
except Exception as e:
    print(f"KG-based meta unavailable ({type(e).__name__}: {e}); "
          "GES-Prior will run without LLM priors (var_meta=pair_meta=None).")
    var_meta, pair_meta = None, None

var_meta


2026-09-23 10:24:12,360 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:24:13,943 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:24:16,651 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:24:20,850 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:24:22,206 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


...

2026-09-23 10:24:24,762 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:24:26,589 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:24:28,048 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


var_meta: 9 columns | pair_meta: 36 pairs (from https://query.wikidata.org/sparql)


{'Hospital.airPollution': 'Air pollution is a human impact on the environment and a subclass of environmental pollution, caused by dust, wildfires, and volcanic eruptions, and causing smog, respiratory disease, and climate change.',
 'Hospital.careQuality': "'Hospital.careQuality' refers to health care quality, a subclass of quality that is a facet of health care.",
 'Hospital.region': 'A region is a geographical area studied by fields like regional geography and area studies, characterized by a geographic location and boundary, and exemplified by places such as the Caribbean, the Sahara Desert, and the far side of the Moon.',
 'Patient.age': "The real-world entity is a person's age, a physical quantity and duration measured in years, derived from their date of birth and part of their lifetime.",
 'Patient.geneticRisk': 'The entity is a type of gene (a nucleic acid structure) that is heritable and part of a homologous gene cluster.',
 'Patient.smoking': "'Patient.smoking' refers to tob

## 3. Causal graph discovery — constrained vs. unconstrained

Same six algorithms as notebook 02 (Plan 1's Assumption-1 hard constraint vs. none), run directly
against the endpoint-backed `ctx`. If the endpoint happens to be serving the same synthetic-clinic
KG `causalway.synthetic.generate()` produces (same class/property names), the known 11-edge ground
truth is used to score precision/recall/SHD too; otherwise the table falls back to structural-only
metrics (edge count, topological validity) — this is a live external endpoint, so the notebook
doesn't assume its content.

In [6]:
from causalway import synthetic as syn
from causalway.result import OntologicalCausalGraph

HAVE_TRUTH = all(c in ctx.column_names and e in ctx.column_names for c, e in syn.GROUND_TRUTH_EDGES)
truth_ocg = None
if HAVE_TRUTH:
    truth_adj = syn.truth_adjacency(ctx.column_names)
    truth_ocg = OntologicalCausalGraph(nodes=ctx.nodes_kept, adj=truth_adj)
    print(f"endpoint node names match causalway.synthetic's ground truth -- "
          f"{len(syn.GROUND_TRUTH_EDGES)} ground-truth edges available for scoring.")
else:
    print("endpoint node names don't match causalway.synthetic's ground truth graph -- "
          "reporting structural metrics only (no precision/recall/SHD against a truth graph).")


endpoint node names match causalway.synthetic's ground truth -- 11 ground-truth edges available for scoring.


In [7]:
from runners.run_kg_discovery import ALLOWED_METHODS

methods = ALLOWED_METHODS  # ['GES', 'GES-Prior', 'PC', 'NOTEARS', 'DAGMA', 'LiNGAM', 'DAG-GNN']

rows = []
adjs = {}
for m in methods:
    for constrained in (True, False):
        # var_meta/pair_meta/domain are only used by GES-Prior; run_algorithm ignores
        # them for every other method.
        adj = run_algorithm(m, ctx, constrained=constrained,
                            var_meta=var_meta, pair_meta=pair_meta, domain=domain)
        adjs[(m, constrained)] = adj
        ocg = OntologicalCausalGraph.from_discovery(adj, ctx.constraint_kept)
        met = {'n_edges': int(adj.sum()),
               'topological_validity': ocg.topological_validity(ctx.constraint_kept)}
        if truth_ocg is not None:
            met.update(ocg.metrics(truth_ocg))
        met['method'] = m
        met['constrained'] = constrained
        rows.append(met)

table = pd.DataFrame(rows).set_index(['method', 'constrained'])
cols = [c for c in ['shd', 'precision', 'recall', 'f1', 'topological_validity', 'n_edges'] if c in table.columns]
table[cols].round(3)


LLM queries:   0%|          | 0/46 [00:00<?, ?pair/s]

2026-09-23 10:25:29,635 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:25:29,636 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:25:29,637 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


2026-09-23 10:25:29,638 - /Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/httpx/_client.py[line:1025] - INFO: HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


...

2026-09-23 10:44:57,353 - /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/algs/dag_gnn.py[line:261] - INFO: Iter: 6, epoch: 299, h_new: 2.01340493433122e-06


2026-09-23 10:46:43,303 - /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/algs/dag_gnn.py[line:261] - INFO: Iter: 7, epoch: 299, h_new: 1.1247360021116037e-07


2026-09-23 10:47:53,455 - /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/algs/dag_gnn.py[line:261] - INFO: Iter: 8, epoch: 299, h_new: 1.9936610584636583e-08


2026-09-23 10:49:38,542 - /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/algs/dag_gnn.py[line:261] - INFO: Iter: 9, epoch: 299, h_new: 1.593610576833271e-09


shd  precision  recall     f1  topological_validity  \
method    constrained                                                        
GES       True           5      0.688   1.000  0.815                   1.0   
          False          5      0.688   1.000  0.815                   1.0   
GES-Prior True           4      0.769   0.909  0.833                   1.0   
          False          2      0.846   1.000  0.917                   1.0   
PC        True          10      0.533   0.727  0.615                   1.0   
          False          9      0.562   0.818  0.667                   1.0   
NOTEARS   True          11      0.500   0.182  0.267                   1.0   
          False         11      0.500   0.182  0.267                   1.0   
DAGMA     True          14      0.286   0.182  0.222                   1.0   
          False         14      0.286   0.182  0.222                   1.0   
LiNGAM    True          16      0.143   0.091  0.111                   1.0   
          False         16      0.143   0.091  0.111                   1.0   
DAG-GNN   True          14      0.200   0.091  0.125                   1.0   
          False         14      0.286   0.182  0.222                   1.0   

                       n_edges  
method    constrained           
GES       True              16  
          False             16  
GES-Prior True              13  
          False             13  
PC        True              15  
          False             16  
NOTEARS   True               4  
          False              4  
DAGMA     True               7  
          False              7  
LiNGAM    True               7  
          False              7  
DAG-GNN   True               5  
          False              7

## 4. Pick a discovered graph and export it

In [8]:
best_key = table['f1'].idxmax() if truth_ocg is not None else ('GES', True)
# idxmax hands the key back from a MultiIndex, so the flag is a numpy.bool_; the RDF
# export types parameters by Python type and would otherwise write it as JSON text.
best_method, best_constrained = best_key[0], bool(best_key[1])
best_adj = adjs[best_key]
print('selected discovery result:', best_method, '| constrained =', best_constrained)

ocg_discovered = to_ocg(best_adj, ctx, best_constrained, method=best_method,
                        params={'constrained': best_constrained}, source=ENDPOINT)

out_dir = os.path.join(PROJECT_ROOT, 'results', 'endpoint')
os.makedirs(out_dir, exist_ok=True)
tag = 'constrained' if best_constrained else 'unconstrained'
ocg_path = os.path.join(out_dir, f'{best_method}_{tag}.ttl')
ocg_discovered.to_turtle(ocg_path)
print('discovered graph saved to', ocg_path)
ocg_discovered.to_dataframe()


selected discovery result: GES-Prior | constrained = False


discovered graph saved to /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/results/endpoint/GES-Prior_unconstrained.ttl


,method,cause,effect,cause_domain,effect_domain,relation,weight
0,GES-Prior,Hospital.airPollution,Hospital.careQuality,Hospital,Hospital,ε,1.0
1,GES-Prior,Hospital.airPollution,Patient.tumorStage,Hospital,Patient,treatedAt (inverse),1.0
2,GES-Prior,Hospital.careQuality,Patient.survival,Hospital,Patient,treatedAt (inverse),1.0
3,GES-Prior,Hospital.region,Hospital.airPollution,Hospital,Hospital,ε,1.0
4,GES-Prior,Hospital.region,Hospital.careQuality,Hospital,Hospital,ε,1.0
5,GES-Prior,Patient.age,Patient.tumorStage,Patient,Patient,ε,1.0
6,GES-Prior,Patient.geneticRisk,Patient.tumorStage,Patient,Patient,ε,1.0
7,GES-Prior,Patient.smoking,Patient.tumorStage,Patient,Patient,ε,1.0
8,GES-Prior,Patient.tumorStage,Patient.survival,Patient,Patient,ε,1.0
9,GES-Prior,Patient.tumorStage,Therapy.dosage,Patient,Therapy,receives (forward),1.0


## 5. Fit the causal model directly against the endpoint

`CausalModel.fit` takes the causal graph and the KG independently (Plan 2 §3.3) — passing
`ENDPOINT` as `kg=` re-resolves the schema and re-materialises the flat join straight from
GraphDB, aligning nodes to `ocg_discovered` on `(domain, prop, range)` identity rather than
reusing `ctx`'s materialization. `on_cycle='weight'` is needed because GES's output isn't
guaranteed acyclic (Plan 1 §3.2); it deliberately breaks any cycle by dropping its lowest-weight
edge instead of raising.

In [9]:
from causalway.model import CausalModel

model = CausalModel.fit(
    ocg_discovered, ENDPOINT, quality='good', random_state=0,
    on_missing='drop', on_cycle='weight',
)
print('model_id:', model.model_id)
model.mechanism_table()


resolve_schema: infer_missing defaults to False for endpoints (pass infer_missing=True to run the T-Box inference queries against it anyway).


Fitting causal models:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.airPollution:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.careQuality:   0%|          | 0/11 [00:00<?, ?it/s] 

Fitting causal mechanism of node Hospital.region:   0%|          | 0/11 [00:00<?, ?it/s]     

...

Fitting causal mechanism of node Therapy.drugClass:  73%|███████▎  | 8/11 [00:05<00:02,  1.10it/s]

Fitting causal mechanism of node Therapy.toxicity:  73%|███████▎  | 8/11 [00:05<00:02,  1.10it/s] 

Fitting causal mechanism of node Therapy.toxicity: 100%|██████████| 11/11 [00:05<00:00,  1.87it/s]

model_id: GES-Prior-f8f537ce75f9-f8f537ce


,node,dtype,is_root,mechanism_type,invertible
0,Hospital.airPollution,categorical,False,InvertibleClassifierFCM,True
1,Hospital.careQuality,categorical,False,InvertibleClassifierFCM,True
2,Hospital.region,categorical,True,EmpiricalDistribution,True
3,Patient.age,categorical,True,EmpiricalDistribution,True
4,Patient.geneticRisk,categorical,True,EmpiricalDistribution,True
5,Patient.smoking,categorical,True,EmpiricalDistribution,True
6,Patient.survival,categorical,False,InvertibleClassifierFCM,True
7,Patient.tumorStage,categorical,False,InvertibleClassifierFCM,True
8,Therapy.dosage,categorical,False,InvertibleClassifierFCM,True
9,Therapy.drugClass,categorical,True,EmpiricalDistribution,True


In [10]:
model_truth = None
if truth_ocg is not None:
    model_truth = CausalModel.fit(truth_ocg, ENDPOINT, quality='good', random_state=0)
    print('model_truth:', model_truth.model_id)
else:
    print('No ground-truth graph available for this endpoint -- skipping the ground-truth model.')


resolve_schema: infer_missing defaults to False for endpoints (pass infer_missing=True to run the T-Box inference queries against it anyway).


Fitting causal models:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.airPollution:   0%|          | 0/11 [00:00<?, ?it/s]

Fitting causal mechanism of node Hospital.careQuality:   0%|          | 0/11 [00:00<?, ?it/s] 

Fitting causal mechanism of node Hospital.region:   0%|          | 0/11 [00:00<?, ?it/s]     

...

Fitting causal mechanism of node Therapy.dosage:  73%|███████▎  | 8/11 [00:06<00:03,  1.03s/it]    

Fitting causal mechanism of node Therapy.drugClass:  73%|███████▎  | 8/11 [00:06<00:03,  1.03s/it]

Fitting causal mechanism of node Therapy.toxicity:  73%|███████▎  | 8/11 [00:06<00:03,  1.03s/it] 

Fitting causal mechanism of node Therapy.toxicity: 100%|██████████| 11/11 [00:06<00:00,  1.67it/s]

model_truth: ocg-4692dd1add79-4692dd1a


## 6. Evaluate before trusting any answer

`held_out_cv` checks that each non-root node's mechanism actually beats a marginal baseline;
`evaluate_model` wraps `gcm.evaluate_causal_model` (mechanism performance + the invertibility
assumption); `falsify` (`gcm.falsify.falsify_graph`) is opt-in — it runs permutation tests over
the flat-join rows and can take a while.

In [11]:
from causalway import evaluation

cv = evaluation.held_out_cv(model, n_splits=5, random_state=0)
cv['graph'] = 'discovered'
frames = [cv]
if model_truth is not None:
    cv_truth = evaluation.held_out_cv(model_truth, n_splits=5, random_state=0)
    cv_truth['graph'] = 'ground_truth'
    frames.append(cv_truth)
pd.concat(frames, ignore_index=True).sort_values(['node', 'graph'])


,node,metric,model,baseline,beats_baseline,macro_f1,graph
0,Hospital.airPollution,accuracy,0.851124,0.455056,True,0.869515,discovered
6,Hospital.airPollution,accuracy,0.851124,0.455056,True,0.869515,ground_truth
1,Hospital.careQuality,accuracy,0.715490,0.664125,True,0.612657,discovered
2,Patient.survival,accuracy,0.736758,0.500602,True,0.696934,discovered
7,Patient.survival,accuracy,0.736758,0.500602,True,0.696934,ground_truth
3,Patient.tumorStage,accuracy,0.678571,0.362761,True,0.651035,discovered
8,Patient.tumorStage,accuracy,0.678571,0.362761,True,0.651035,ground_truth
4,Therapy.dosage,accuracy,0.730337,0.513844,True,0.659873,discovered
9,Therapy.dosage,accuracy,0.730337,0.513844,True,0.659873,ground_truth
5,Therapy.toxicity,accuracy,0.886637,0.674358,True,0.867242,discovered


In [12]:
# eval_result = evaluation.evaluate_model(model)
# print(eval_result)

# RUN_FALSIFY = False  # gcm.falsify.falsify_graph over the flat join; opt-in, can take a while
# if RUN_FALSIFY:
#     print(evaluation.falsify(model, n_permutations=20, show_progress_bar=True))
# else:
#     print('Skipped -- set RUN_FALSIFY = True to run gcm.falsify.falsify_graph.')


## 7. Conditional inference — P(target | evidence)

The running example below is whichever edge `ocg_discovered` put first; `causalway.inference`
picks the cheapest applicable backend automatically (mechanism evaluation → exact pgmpy →
likelihood weighting → rejection) and reports which one it used on `Answer.backend`.

A query has **1..n targets** (Plan 2 §6.3 — `Query.target` was always a list). All three
engines take either one node name or a collection of them, and the shape you ask in is the
shape you get back: a `str` returns one `Answer`, a list/tuple/set returns `{node: Answer}`.
A batch is not a loop — every target is read off *one* shared computation (one pgmpy network,
one weighted sample set), so the answers are mutually consistent and the expensive step is
paid once instead of once per target.

In [13]:
edges_df = ocg_discovered.to_dataframe()
if len(edges_df) == 0:
    raise RuntimeError('The selected discovery result has no edges -- pick a different (method, constrained) key.')

example_cause, example_effect = edges_df.iloc[0][['cause', 'effect']]
print(f'running example edge: {example_cause} -> {example_effect}')

example_value = model.spec.data[example_cause].iloc[0]
ans_cond = model.condition(example_effect, {example_cause: example_value})
print(f'P({example_effect} | {example_cause} = {example_value!r}):')
print(ans_cond.distribution or ans_cond.predicted, f'(backend={ans_cond.backend})')

running example edge: Hospital.airPollution -> Hospital.careQuality
P(Hospital.careQuality | Hospital.airPollution = 'Low'):
{'High': 0.5874720982142857, 'Medium': 0.41252790178571425} (backend=pgmpy-exact)


In [14]:
# The same evidence, asked about several targets at once. `targets` is a set here to make
# the point that any collection works; the answers come back keyed by node name.
targets = {example_effect, *[c for c in model.spec.columns
                             if c not in (example_cause, example_effect)][:2]}
ans_multi = model.condition(targets, {example_cause: example_value})

pd.DataFrame([
    {'target': node,
     'predicted': a.predicted,
     'distribution': a.distribution,
     'backend': a.backend,
     # One shared sample set for the batch: `ess` is identical across targets whenever the
     # answer came from a sampling tier, which is what "they cannot disagree" looks like.
     'ess': a.ess}
    for node, a in ans_multi.items()
])

,target,predicted,distribution,backend,ess
0,Hospital.careQuality,High,"{'High': 0.5874720982142857, 'Medium': 0.41252...",pgmpy-exact,4984.0
1,Hospital.region,North,"{'East': 0.00034877232142857144, 'North': 0.99...",pgmpy-exact,4984.0
2,Patient.age,Young,"{'Middle': 0.32124006146856415, 'Old': 0.32284...",pgmpy-exact,4984.0


## 8. Interventional inference — do(X := x), population-level with a contrast

Same 1..n targets. One `do()` draw is a full joint row, so a batch of targets is read off a
single sample of the mutilated model — and when a `reference=` is given, contrasted against a
single reference world rather than one drawn per target.

In [15]:
values = model.spec.data[example_cause].dropna().unique().tolist()
v_alt = values[0]
v_ref = values[1] if len(values) > 1 else values[0]

ans_interv = model.intervene({example_cause: v_alt}, target=targets,
                             reference={example_cause: v_ref}, num_samples=10_000)
print(f'population do({example_cause} := {v_alt!r}) vs do({example_cause} := {v_ref!r}):')

pd.DataFrame([
    {'target': node,
     'predicted': a.predicted,
     'distribution': a.distribution if a.distribution is not None else a.mean,
     'effect (alt - ref)': a.effect}
    for node, a in ans_interv.items()
])

population do(Hospital.airPollution := 'Low') vs do(Hospital.airPollution := 'High'):


,target,predicted,distribution,effect (alt - ref)
0,Hospital.careQuality,High,"{'High': 0.6096, 'Medium': 0.3904}","{'High': 0.09510000000000007, 'Medium': -0.095..."
1,Hospital.region,East,"{'East': 0.3108, 'West': 0.3037, 'North': 0.24...","{'East': 0.0015000000000000013, 'North': 0.002..."
2,Patient.age,Young,"{'Young': 0.3572, 'Old': 0.328, 'Middle': 0.3148}","{'Middle': -0.007899999999999963, 'Old': 0.005..."


## 9. Counterfactual inference — entity-level

`CausalModel.counterfactual` abducts noise from one entity's own flat-join row(s), then evaluates
the intervention through the fitted mechanisms (Plan 2 §5.3) — a genuine per-entity
counterfactual, not a population contrast.

*Which* rows are this entity's is the whole basis of that claim, and it comes from
`model.spec.mat.entity_ids` — the flat join's record of which entity owns which cell, one
column per class variable, aligned by row index (Plan 1 D1). `population_rows` is the lookup;
`Answer.per_row` hands back the per-row counterfactual values for the same rows. A model
fitted on a *different* join than the one an entity was chosen from would abduct from the
wrong rows, which is why `CausalModel.fit` now accepts the same `excluded` / `excluded_joins`
curation the upstream materialisation used.

Multi-target matters most here: the noise is this entity's, so a second draw per target would
answer "what would have happened to this unit" from a different draw of the unit each time.
Every target below comes from one sequence of abductions.

In [16]:
from causalway.entities import population_rows

cause_var = next(n.var for n in ocg_discovered.nodes if n.name == example_cause)
counts = model.spec.mat.entity_ids[cause_var].value_counts()
example_entity = str(counts.index[0])
rows = population_rows(model.spec.mat, example_entity)
print(f'{example_entity} ({cause_var}) spans {len(rows)} flat-join row(s)')
print('the rows this unit owns, as materialised:')
display(model.spec.data.loc[rows, sorted(targets | {example_cause})])

ans_cf = model.counterfactual({(example_entity, example_cause): v_alt},
                              entity=example_entity, target=targets, num_samples=200)
print(f'counterfactual under do({example_cause} := {v_alt!r}) for {example_entity}:')

pd.DataFrame([
    {'target': node,
     'entity': a.entity,
     'factual (this unit, observed)': model.spec.data.loc[rows, node].mode().iloc[0],
     'counterfactual': a.predicted,
     'distribution': a.distribution if a.distribution is not None else a.mean,
     'coupling': a.coupling,
     'backend': a.backend}
    for node, a in ans_cf.items()
])

http://causalkg.example.org/synthetic/hospital_15 (Hospital) spans 297 flat-join row(s)
the rows this unit owns, as materialised:


,Hospital.airPollution,Hospital.careQuality,Hospital.region,Patient.age
3744,High,High,West,Young
3745,High,High,West,Young
3746,High,High,West,Middle
3747,High,High,West,Middle
3748,High,High,West,Old
...,...,...,...,...
4036,High,High,West,Middle
4037,High,High,West,Middle
4038,High,High,West,Middle
4039,High,High,West,Young


counterfactual under do(Hospital.airPollution := 'Low') for http://causalkg.example.org/synthetic/hospital_15:


,target,entity,"factual (this unit, observed)",counterfactual,distribution,coupling,backend
0,Hospital.careQuality,http://causalkg.example.org/synthetic/hospital_15,High,High,{'High': 1.0},gumbel-max,gumbel-max-abduction
1,Hospital.region,http://causalkg.example.org/synthetic/hospital_15,West,West,{'West': 1.0},gumbel-max,gumbel-max-abduction
2,Patient.age,http://causalkg.example.org/synthetic/hospital_15,Young,Young,"{'Young': 0.3939393939393939, 'Middle': 0.3535...",gumbel-max,gumbel-max-abduction


## 10. Cross-entity reach validation (§6.5)

`validate_query` enforces two checks before an intervention is allowed to answer a query: a
causal path from the intervened node to the target, and — for an intervention naming a
*different* entity than the query's subject — that the two entities are actually joined in the
source KG (relational reach). This only has something to demonstrate when the discovered graph
has at least one cross-class edge; skipped cleanly otherwise.

In [17]:
from causalway.entities import rows_for_entity
from causalway.queries import Intervention, Query, validate_query

cross_rows = edges_df[edges_df['cause_domain'] != edges_df['effect_domain']]
if len(cross_rows) == 0:
    print('No cross-class edge in the discovered graph -- skipping the cross-entity reach demo.')
else:
    r = cross_rows.iloc[0]
    cross_cause, cross_effect = r['cause'], r['effect']
    cause_var2 = next(n.var for n in ocg_discovered.nodes if n.name == cross_cause)
    effect_var2 = next(n.var for n in ocg_discovered.nodes if n.name == cross_effect)

    effect_entity = str(model.spec.mat.entity_ids[effect_var2].iloc[0])
    own_row = rows_for_entity(model.spec.mat, effect_entity, var=effect_var2)[0]
    own_cause_entity = str(model.spec.mat.entity_ids.loc[own_row, cause_var2])
    other_cause_entity = next(
        e for e in model.spec.mat.entity_ids[cause_var2].astype(str).unique()
        if e != own_cause_entity
    )
    cause_value = model.spec.data.loc[own_row, cross_cause]

    q_own = Query(kind='counterfactual', model_id=model.model_id, target=[cross_effect],
                 interventions=[Intervention(node=cross_cause, value=cause_value, entity=own_cause_entity)],
                 entity=effect_entity)
    validate_query(q_own, model.spec.ocg, model.spec.mat)
    print(f"validate_query accepted {effect_entity}'s own {cause_var2} ✓")

    q_other = Query(kind='counterfactual', model_id=model.model_id, target=[cross_effect],
                    interventions=[Intervention(node=cross_cause, value=cause_value, entity=other_cause_entity)],
                    entity=effect_entity)
    try:
        validate_query(q_other, model.spec.ocg, model.spec.mat)
        print(f'unexpectedly accepted an unrelated {cause_var2}')
    except ValueError as e:
        print(f'validate_query correctly rejected an unrelated {cause_var2}:', e)

    ans_spillover = model.counterfactual({(own_cause_entity, cross_cause): cause_value},
                                         entity=effect_entity, target=cross_effect, num_samples=100)
    print()
    print(f'counterfactual {cross_effect} for {effect_entity} '
          f'under do({cross_cause} := {cause_value!r} via its own {cause_var2}):',
          ans_spillover.distribution if ans_spillover.distribution is not None else ans_spillover.predicted)


validate_query accepted http://causalkg.example.org/synthetic/patient_13's own Hospital ✓
validate_query correctly rejected an unrelated Hospital: validate_query: http://causalkg.example.org/synthetic/hospital_1 is not joined to http://causalkg.example.org/synthetic/patient_13 in the source KG along any relation (§6.5 relational reach).

counterfactual Patient.tumorStage for http://causalkg.example.org/synthetic/patient_13 under do(Hospital.airPollution := 'Low' via its own Hospital): {'II': 1.0}


## 11. Persist the fitted model(s)

In [18]:
for m in [model, model_truth]:
    if m is None:
        continue
    model_dir = os.path.join(PROJECT_ROOT, 'results', 'models', m.model_id)
    m.save(model_dir)
    print('saved', m.model_id, '->', model_dir)


saved GES-Prior-f8f537ce75f9-f8f537ce -> /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/results/models/GES-Prior-f8f537ce75f9-f8f537ce


saved ocg-4692dd1add79-4692dd1a -> /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/results/models/ocg-4692dd1add79-4692dd1a


## Limitations carried over from notebooks 02/04

- **Flat-join i.i.d. violation** (Plan 1 §9.4 / Plan 2 §10 risk #4): a 1:N relation duplicates
  rows, so every metric above is computed over a row set with duplicated entities, not i.i.d.
  samples — a documented limitation, not something this notebook hides.
- **`CausalModel.fit(kg=ENDPOINT)` re-queries GraphDB independently** of `ctx` (built in §2): the
  two materializations use the same `resolve_schema`/`materialize` code path but are not
  guaranteed to be pixel-identical if the endpoint's data changes between cells.
- If §3's `HAVE_TRUTH` came back `False`, every metric in this notebook is descriptive
  (topological validity, mechanism cross-validation against a marginal baseline) rather than
  scored against a known generative process — unlike notebook 04, which always has the synthetic
  SEM to check against.